In [ ]:
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, losses, metrics, datasets
import tensorflow_hessian as tfh

print("TensorFlow version:", tf.__version__)

(train_X, train_t), (test_X, test_t) = datasets.mnist.load_data()
X = tf.convert_to_tensor(train_X, dtype=tf.float32) / 255.0
t = tf.convert_to_tensor(train_t, dtype=tf.int32)

class MyModel(tfh.models.Model):
    def __init__(self):
        super().__init__()
        self.flatten = layers.Flatten()
        self.dense1 = tfh.layers.Dense(15)
        self.leaky_relu = layers.LeakyReLU()
        self.dense2 = tfh.layers.Dense(10)
        self.softmax = layers.Activation('softmax')

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.leaky_relu(x)
        x = self.dense2(x)
        x = self.softmax(x)
        return x

model = MyModel()
optimizer = tfh.optimizers.NewtonMethod(eta=0.1, alpha=1.0)

BATCH_SIZE = 2048
train_dataset = tf.data.Dataset.from_tensor_slices((X, t)).batch(BATCH_SIZE)

def step(X, t, training=True):

    if training:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape2:
                y = model(X, training=training)
                loss = losses.SparseCategoricalCrossentropy()(t, y)
            grads = tape2.gradient(loss, model.trainable_variables)
        hessians = tape1.jacobian(grads[0], model.trainable_variables)
        optimizer.apply_gradients(model.trainable_variables, grads, hessians)
    else:
        y = model(X, training=training)
        loss = losses.SparseCategoricalCrossentropy()(t, y)

    accuracy = metrics.SparseCategoricalAccuracy()(t, y)
    return loss, accuracy

for epoch in range(10):

    total_loss = 0.0
    total_accuracy = 0.0
    total_data = 0
    for X, t in (pb := tqdm(train_dataset, desc=f"Epoch {epoch+1}")):
        loss, accuracy = step(X, t, training=True)

        total_loss += loss.numpy() * X.shape[0] 
        total_accuracy += accuracy.numpy() * X.shape[0]
        total_data += X.shape[0]
        pb.set_postfix({"loss": total_loss / total_data, "accuracy": total_accuracy / total_data})

2025-10-06 14:32:24.780961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759728744.790925 1157497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759728744.793805 1157497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759728744.802428 1157497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759728744.802439 1157497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759728744.802440 1157497 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0


I0000 00:00:1759728746.975529 1157497 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
Epoch 1:   0%|          | 0/30 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1759728751.595594 1157497 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0xbd98a70
Epoch 1:  13%|█▎        | 4/30 [00:15<01:36,  3.72s/it, loss=478, accuracy=0.108] 

Epoch 1:  17%|█▋        | 5/30 [00:18<01:31,  3.68s/it, loss=524, accuracy=0.107]

Epoch 10: 100%|██████████| 30/30 [01:46<00:00,  3.56s/it, loss=3.42, accuracy=0.798]


In [1]:
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, losses, metrics, datasets, optimizers
import tensorflow_hessian as tfh

print("TensorFlow version:", tf.__version__)

(train_X, train_t), (test_X, test_t) = datasets.mnist.load_data()
X = tf.convert_to_tensor(train_X, dtype=tf.float32) / 255.0
t = tf.convert_to_tensor(train_t, dtype=tf.int32)

class MyModel(tfh.models.Model):
    def __init__(self):
        super().__init__()
        self.flatten = layers.Flatten()
        self.dense1 = tfh.layers.Dense(15)
        self.leaky_relu = layers.LeakyReLU()
        self.dense2 = tfh.layers.Dense(10)
        self.softmax = layers.Activation('softmax')

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.leaky_relu(x)
        x = self.dense2(x)
        x = self.softmax(x)
        return x

model = MyModel()
optimizer = optimizers.Adam(learning_rate=0.01)

BATCH_SIZE = 2048
train_dataset = tf.data.Dataset.from_tensor_slices((X, t)).batch(BATCH_SIZE)

def step(X, t, training=True):

    if training:
        
        with tf.GradientTape() as tape2:
            y = model(X, training=training)
            loss = losses.SparseCategoricalCrossentropy()(t, y)
        grads = tape2.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
    else:
        y = model(X, training=training)
        loss = losses.SparseCategoricalCrossentropy()(t, y)

    accuracy = metrics.SparseCategoricalAccuracy()(t, y)
    return loss, accuracy

for epoch in range(10):

    total_loss = 0.0
    total_accuracy = 0.0
    total_data = 0
    for X, t in (pb := tqdm(train_dataset, desc=f"Epoch {epoch+1}")):
        loss, accuracy = step(X, t, training=True)

        total_loss += loss.numpy() * X.shape[0] 
        total_accuracy += accuracy.numpy() * X.shape[0]
        total_data += X.shape[0]
        pb.set_postfix({"loss": total_loss / total_data, "accuracy": total_accuracy / total_data})

2025-10-07 22:13:29.425499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759842809.477483     841 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759842809.491834     841 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759842809.618171     841 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759842809.618192     841 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759842809.618193     841 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0


I0000 00:00:1759842813.738804     841 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
Epoch 10: 100%|██████████| 30/30 [00:00<00:00, 77.42it/s, loss=0.57, accuracy=0.85]  


In [2]:
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, losses, metrics, datasets
import tensorflow_hessian as tfh

print("TensorFlow version:", tf.__version__)

(train_X, train_t), (test_X, test_t) = datasets.mnist.load_data()
X = tf.convert_to_tensor(train_X, dtype=tf.float32)[..., tf.newaxis] / 255.0
t = tf.convert_to_tensor(train_t, dtype=tf.int32)

class MyModel(tfh.models.Model):
    def __init__(self):
        super().__init__()
        self.conv1 = tfh.layers.Conv2D(4, kernel_size=(3, 3), padding='same')
        self.leaky_relu1 = layers.LeakyReLU()
        self.gap = layers.GlobalAveragePooling2D()
        self.flatten = layers.Flatten()
        self.leaky_relu2 = layers.LeakyReLU()
        self.dense2 = tfh.layers.Dense(10)
        self.softmax = layers.Activation('softmax')

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.leaky_relu1(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.leaky_relu2(x)
        x = self.dense2(x)
        x = self.softmax(x)
        return x

model = MyModel()
optimizer = tfh.optimizers.MomentumNewtonMethod(eta=0.1, mu = 0.5, alpha=0.5)

BATCH_SIZE = 512
train_dataset = tf.data.Dataset.from_tensor_slices((X, t)).batch(BATCH_SIZE)

def step(X, t, training=True):

    if training:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape2:
                y = model(X, training=training)
                loss = losses.SparseCategoricalCrossentropy()(t, y)
            grads = tape2.gradient(loss, model.trainable_variables)
        hessians = tape1.jacobian(grads[0], model.trainable_variables)
        optimizer.apply_gradients(model.trainable_variables, grads, hessians)
    else:
        y = model(X, training=training)
        loss = losses.SparseCategoricalCrossentropy()(t, y)

    accuracy = metrics.SparseCategoricalAccuracy()(t, y)
    return loss, accuracy

for epoch in range(10):

    total_loss = 0.0
    total_accuracy = 0.0
    total_data = 0
    for X, t in (pb := tqdm(train_dataset, desc=f"Epoch {epoch+1}")):
        loss, accuracy = step(X, t, training=True)

        total_loss += loss.numpy() * X.shape[0] 
        total_accuracy += accuracy.numpy() * X.shape[0]
        total_data += X.shape[0]
        pb.set_postfix({"loss": total_loss / total_data, "accuracy": total_accuracy / total_data})

TensorFlow version: 2.19.0


Epoch 1:   0%|          | 0/118 [00:00<?, ?it/s]I0000 00:00:1759842831.576351     841 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1759842833.319359     841 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x31d9f440
Epoch 1:   3%|▎         | 4/118 [00:04<02:10,  1.14s/it, loss=2.62, accuracy=0.0957]

Epoch 1:   4%|▍         | 5/118 [00:05<02:03,  1.10s/it, loss=2.61, accuracy=0.0969]

Epoch 6:  91%|█████████ | 107/118 [01:53<00:11,  1.04s/it, loss=2.11, accuracy=0.218]2025-10-07 22:26:13.872964: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 551.25MiB (rounded to 578027520)requested by op gradient_tape/Conv2D/pfor/TensorArrayV2Stack/TensorListStack
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-10-07 22:26:13.873858: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-10-07 22:26:13.873887: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 7707, Chunks in use: 7706. 1.88MiB allocated for chunks. 1.88MiB in use in bin. 739.0KiB client-requested in use in bin.
2025-10-07 22:26:13.873895: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin

ResourceExhaustedError: Graph execution error:

Detected at node gradient_tape/Conv2D/pfor/TensorArrayV2Stack/TensorListStack defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/user/colab_20250722/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>

  File "/home/user/colab_20250722/lib/python3.12/site-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/home/user/colab_20250722/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 712, in start

  File "/home/user/colab_20250722/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/user/colab_20250722/lib/python3.12/si